In [14]:
## Step 1 - Imports ##

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, MaxPooling1D, Flatten

In [15]:
## Step 2 - Load the data ##

# Load well datasets
well1 = pd.read_csv("1_2021002_.csv")
well4 = pd.read_csv("4_20211022_.csv")
well7 = pd.read_csv("7_2021022_.csv")

In [16]:
## Step 3 - Clean and combine data ##


# Fix typo
well4 = well4.rename(columns={"litholody": "lithology"})
well7 = well7.rename(columns={"litholody": "lithology"})

# Combine all wells
all_data = pd.concat([well1, well4, well7], axis=0)

# Features
features = ["SP", "GR", "LLD", "LLS", "DEN", "AC"]

all_data = all_data[features + ["lithology"]]

# Remove missing
all_data = all_data.dropna()

In [17]:
## Step 4 - Define X and y ##

X = all_data[features]
y = all_data["lithology"]

In [18]:
## Step 5 - Encode labels ##

le = LabelEncoder()
y = le.fit_transform(y)

In [19]:
## Step 6 - Random train/test split ##


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [20]:
## Step 7 - Fill missing values ##

X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_train.mean())

In [21]:
## Step 8 - Reshape for CNN  ##

X_train = X_train.values.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.values.reshape(X_test.shape[0], X_test.shape[1], 1)

In [22]:
## Step 9 - Build CNN model ##

model = Sequential()

model.add(Conv1D(64, 2, activation='relu', input_shape=(6,1)))
model.add(MaxPooling1D(2))
model.add(Flatten())

model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [23]:
## Step 10 - Compile model ##

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [24]:
## Step 11 - Train model ##

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3974 - accuracy: 0.7989
Epoch 2/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3536 - accuracy: 0.8259
Epoch 3/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3440 - accuracy: 0.8334
Epoch 4/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3397 - accuracy: 0.8387
Epoch 5/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3450 - accuracy: 0.8342
Epoch 6/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3316 - accuracy: 0.8471
Epoch 7/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3344 - accuracy: 0.8420
Epoch 8/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3266 - accuracy: 0.8495
Epoch 9/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3268 - accuracy: 0.8493
Epoch 10/20
333/333 [==============================] - 1s 2ms/step - loss: 0.3240 - accuracy: 0.8501

In [26]:
## Step 12 - Evaluate model ##

loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy", accuracy)

167/167 [==============================] - 2s 1ms/step - loss: 0.3010 - accuracy: 0.8668
Test Accuracy 0.8667543530464172


In [28]:
## Step 13 - Interpetations ##

#The Convolutional Neural Network (CNN) model was trained using a randomly split dataset combining all wells. The model achieved an accuracy of approximately 86.7%, which is higher than expected for deep learning models applied to tabular data. This improved performance is primarily due to the use of a random train–test split, allowing the model to learn patterns across all wells. However, this approach introduces some level of data leakage, as samples from the same well may appear in both training and testing datasets. Despite this, the results demonstrate that CNNs can effectively learn complex relationships in well log data. In comparison, tree-based models such as Random Forest and Decision Tree still provide slightly higher accuracy while being more interpretable.
